# Chinese-LLaMA-Alpaca

## 导包
- `build_instruction_dataset`负责加载和构建训练模型所用的数据集。该函数可能会从原始数据源加载数据、进行必要的预处理，并以适当的格式返回数据。
- `load_dataset`用于从各种数据源加载数据集，支持本地文件和在线资源加载。通过datasets库中的函数和工具，为模型训练和评估作准备。
- `concatenate_datasets`负责将多个数据集对象进行合并操作，通常用于将多个分开的数据集整合为一个以便进行训练。
- `AutoConfig`用于自动加载模型的配置，通过模型名称识别和加载。
- `AutoModelForCausalLM`负责提供基于自回归的语言模型，通常用于生成任务。
- `LlamaTokenizer`和`AutoTokenizer`用于文本的分词处理，这对于将文本数据转换为模型可理解的输入格式至关重要。
- `Trainer`类用于封装训练过程，把各种训练要素（数据、模型、优化器等）结合起来进行训练。
- `TrainingArguments`用于配置训练过程中的各种参数，如学习率、批次大小等。
- `accuracy_score`用来计算模型预测结果的准确率，是评估分类模型性能的重要指标。
- `LoraConfig`和`TaskType`与PEFT框架相关，通常用于特定任务的模型配置优化，帮助在小型数据和低资源环境下训练高效的模型。


In [ ]:
import logging
import math
import os
import sys

import numpy as np
from typing import Optional, List, Dict, Any, Mapping
from itertools import chain
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path
import datasets
import torch
from build_dataset import build_instruction_dataset, DataCollatorForSupervisedDataset
import transformers
from datasets import load_dataset, concatenate_datasets
from transformers import BitsAndBytesConfig
from transformers import (
    CONFIG_MAPPING,
    AutoConfig,
    AutoModelForCausalLM,
    LlamaForCausalLM,
    LlamaTokenizer,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    set_seed,
    AutoTokenizer,
    LlamaTokenizer
)
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import send_example_telemetry
from transformers.utils.versions import require_version

from sklearn.metrics import accuracy_score
from peft import LoraConfig, TaskType, get_peft_model, PeftModel, get_peft_model_state_dict
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR


# 配置文本数据处理的特殊标记及环境依赖

这段代码主要配置了在文本数据处理中需要使用的一些特殊标记，并且配置了一个环境依赖的检查。这些标记通常在自然语言处理（NLP）模型中，用于指示文本序列的不同用途或状态。环境依赖检测用于确保需要的软件库存在并满足最低版本要求。

## 代码作用

### 特殊标记的定义
这段代码定义了一些用于文本数据处理的特殊标记：

- `IGNORE_INDEX = -100`: 该常量通常在损失计算和指标计算中使用，用于忽略某些特定位置上的标签，比如在计算交叉熵损失时忽略的填充标记的位置。
- `DEFAULT_PAD_TOKEN = "[PAD]"`: PAD标记用于将文本序列填充到相同的长度。这在批处理中尤其重要，以确保所有序列的长度一致。
- `DEFAULT_EOS_TOKEN = "</s>"`: EOS（End Of Sequence）标记用于指示序列的结束。这通常用于生成任务或序列到序列任务中，帮助模型辨识序列终点。
- `DEFAULT_BOS_TOKEN = "<s>"`: BOS（Beginning Of Sequence）标记用于指示序列的开始，常用于序列生成任务中。
- `DEFAULT_UNK_TOKEN = "<unk>"`: UNK（Unknown）标记用于替代词汇表中不存在的词汇，在文本数据预处理中，处理未知词需要这个标记。

### 环境依赖检测
```python
require_version("datasets>=1.8.0", "To fix: pip install -r examples/pytorch/language-modeling/requirements.txt")
```
该代码行用于确保“datasets”库的版本不低于1.8.0。若版本不符合要求，提示用户通过指定命令（`pip install -r examples/pytorch/language-modeling/requirements.txt`）来安装正确的依赖。

In [ ]:
IGNORE_INDEX = -100
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_UNK_TOKEN = "<unk>"

require_version("datasets>=1.8.0", "To fix: pip install -r examples/pytorch/language-modeling/requirements.txt")
"""
require_version(version_constraint, error_message)
Args:
    version_constraint (str): 表示版本约束的字符串，例如 "datasets>=1.8.0"。
    error_message (str): 当版本不满足要求时显示的错误信息。

作用:
    检查给定软件包的版本是否满足最低要求。
    如果不满足要求，抛出错误并提示用户如何修复。

Returns:
    None

"""

### `SavePeftModelCallback`

`SavePeftModelCallback` 是一个针对 `transformers.TrainerCallback` 的回调类，用于在特定时刻保存预训练模型和分词器，使得在训练过程中能有效地记录和保存最优的模型检查点和最终模型。

### 注意
- 动态路径管理，在保存模型过程中，需要根据当前的模型状态动态确定保存路径，这涉及对路径字符串的管理和拼接。
- 数据类型转换，在处理不同类型数据时，需要准确地进行数据类型转换，例如 `torch.Tensor` 和 `np.ndarray` 的处理及类型推断。
- 模型输出的后处理，`preprocess_logits_for_metrics` 函数需要对模型的logits进行变换，处理可能存在的logits额外维度。此过程涉及对模型框架和输出格式的了解。
- 容错数据批次组装，在构造批次数据时，需要处理不同特征的数据类型及其可能的缺失或不一致，这要求开发者对数据处理逻辑有较深的理解。

In [ ]:
class SavePeftModelCallback(transformers.TrainerCallback):
    def save_model(self, args, state, kwargs):
        """
            保存当前模型和分词器。
            
            参数:
            args: 训练参数。
            state: 当前训练状态，包括步骤和检查点信息。
            kwargs: 其他参数字典，包含模型和分词器对象。
        """
        if state.best_model_checkpoint is not None: # 管理模型检查点保存路径
            checkpoint_folder = os.path.join(state.best_model_checkpoint, "pt_lora_model")
        else:
            checkpoint_folder = os.path.join(args.output_dir, f"{PREFIX_CHECKPOINT_DIR}-{state.global_step}")
        # 保存模型与分词器到指定路径
        peft_model_path = os.path.join(checkpoint_folder, "pt_lora_model")
        kwargs["model"].save_pretrained(peft_model_path)
        kwargs["tokenizer"].save_pretrained(peft_model_path)

    def on_save(self, args, state, control, **kwargs):
        """在训练过程中周期性调用，用于保存检查点"""
        self.save_model(args, state, kwargs)
        return control

    def on_train_end(self, args, state, control, **kwargs):
        """在训练结束时调用，用于保存最终的模型和分词器。"""
        peft_model_path = os.path.join(args.output_dir, "pt_lora_model")
        kwargs["model"].save_pretrained(peft_model_path)
        kwargs["tokenizer"].save_pretrained(peft_model_path)

In [ ]:




def accuracy(predictions, references, normalize=True, sample_weight=None):
    """
        用于计算模型的预测准确率。
        
        参数:
        predictions: 模型预测结果。
        references: 数据标签。
        normalize: 是否返回归一化后的结果。
        sample_weight: 样本权重。
    """
    return {
        "accuracy": float(
            accuracy_score(references, predictions, normalize=normalize, sample_weight=sample_weight)
        )
    }


def compute_metrics(eval_preds):
    """
        用于处理预测结果，以计算准确率为指标。
        
        参数:
        eval_preds: 模型的预测结果和真实标签。
        
        返回:
        包含准确率的字典。
    """
    preds, labels = eval_preds
    # preds have the same shape as the labels, after the argmax(-1) has been calculated
    # by preprocess_logits_for_metrics but we need to shift the labels
    labels = labels[:, 1:].reshape(-1)
    preds = preds[:, :-1].reshape(-1)
    return accuracy(predictions=preds, references=labels)


def preprocess_logits_for_metrics(logits, labels):
    """
        预处理模型输出的logits以用于指标计算。
        
        参数:
        logits: 模型输出的logits。
        labels: 标签数据。
    """
    if isinstance(logits, tuple):
        # Depending on the model and config, logits may contain extra tensors,
        # like past_key_values, but logits always come first
        logits = logits[0]
    return logits.argmax(dim=-1)


def fault_tolerance_data_collator(features: List) -> Dict[str, Any]:
    """
        该函数用于组装数据批次，以在训练过程中处理数据集的容错性。
        
        参数:
        features: 输入数据特征的列表。
        
        返回:
        数据批次字典。
    """
    if not isinstance(features[0], Mapping):
        features = [vars(f) for f in features]
    first = features[0]
    batch = {}

    # Special handling for labels.
    # Ensure that tensor is created with the correct type
    # (it should be automatically the case, but let's make sure of it.)
    if "label" in first and first["label"] is not None:
        label = first["label"].item() if isinstance(first["label"], torch.Tensor) else first["label"]
        dtype = torch.long if isinstance(label, int) else torch.float
        batch["labels"] = torch.tensor([f["label"] for f in features], dtype=dtype)
    elif "label_ids" in first and first["label_ids"] is not None:
        if isinstance(first["label_ids"], torch.Tensor):
            batch["labels"] = torch.stack([f["label_ids"] for f in features])
        else:
            dtype = torch.long if isinstance(first["label_ids"][0], int) else torch.float
            batch["labels"] = torch.tensor([f["label_ids"] for f in features], dtype=dtype)

    # Handling of all other possible keys.
    # Again, we will use the first element to figure out which key/values are not None for this model.

    try:
        for k, v in first.items():
            if k not in ("label", "label_ids") and v is not None and not isinstance(v, str):
                if isinstance(v, torch.Tensor):
                    batch[k] = torch.stack([f[k] for f in features])
                elif isinstance(v, np.ndarray):
                    batch[k] = torch.tensor(np.stack([f[k] for f in features]))
                else:
                    batch[k] = torch.tensor([f[k] for f in features])
    except ValueError: # quick fix by simply take the first example
        for k, v in first.items():
            if k not in ("label", "label_ids") and v is not None and not isinstance(v, str):
                if isinstance(v, torch.Tensor):
                    batch[k] = torch.stack([features[0][k]] * len(features))
                elif isinstance(v, np.ndarray):
                    batch[k] = torch.tensor(np.stack([features[0][k]] * len(features)))
                else:
                    batch[k] = torch.tensor([features[0][k]] * len(features))

    return batch


# MODEL_CONFIG_CLASSES = list(MODEL_FOR_CAUSAL_LM_MAPPING.keys())
# MODEL_TYPES = tuple(conf.model_type for conf in MODEL_CONFIG_CLASSES)


# LlamaTokenizer的加载与初始化

下面这段代码加载`ZiQingYang/chinese-llama-lora-7b`的预训练分词器，并初始化。

**参数:**
- `pretrained_model_name_or_path`: 类型为`str`，指定要加载的预训练分词器的名称或路径。
- `truth_remote`: 类型为`bool`，可选参数，默认为`True`。该参数通常与分布式训练或模型缓存策略相关，具体行为依照实现情况确定。**需要注意的是，该参数可能是自定义实现的一部分，因为经典的Hugging Face `transformers`库中没有`truth_remote`参数，因此这个参数需要查看特定实现的文档或代码以了解其实际作用。**


In [ ]:
tokenizer_name = 'ziqingyang/chinese-llama-lora-7b'
tokenizer = LlamaTokenizer.from_pretrained(tokenizer_name, truth_remote = True)

# 文本预处理和数据集准备
下面这段代码的主要作用是加载文本数据集，然后进行分词和文本分组，并将处理后的数据集保存到磁盘中。

In [ ]:
block_size = 256
torch_dtype = 'fp16'

# model_name = "TinyPixel/Llama-2-7B-bf16-sharded"
model_name = 'HuggingFaceM4/tiny-random-LlamaForCausalLM'


### 数据集分词
使用 `tokenizer` 对输入文本进行分词，将文本转换为相应的标记序列。

In [ ]:

def tokenize_function(examples):
    """对给定的文本进行分词处理

    Args:
        examples (dict): 包含文本数据的字典。
    
    Returns:
        output: 分词后的结果。
    """
    output = tokenizer(examples["text"])
    return output

### 文本分组
- 通过 `chain(*examples[k])` 将多个文本序列连接在一起，然后按 `block_size` 分组。
- 每个分组后结果的“标签”与原文本相同，这是为语言模型的训练准备。

In [ ]:
def group_texts(examples):
    """将分词后的文本分组，形成固定大小的文本块。
        Args:
            examples (dict): 包含分词数据的字典。
        
        Returns:
            result: 分组后的文本块结果。
    """
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
dataset_dir = './../../data'
data_cache_dir = 'temp_data_cache_dir'
lm_datasets = []
path = Path(dataset_dir) # 通过 `Path().glob("*.txt")` 列出数据目录中的所有文本文件，准备依次加载。

files = [file.name for file in path.glob("*.txt")]
for idx, file in enumerate(files):
    data_file = os.path.join(path, file)
    filename = ''.join(file.split(".")[:-1])
    cache_path = os.path.join(data_cache_dir, filename)
    os.makedirs(cache_path, exist_ok=True)
    if True:
        cache_dir = os.path.join(data_cache_dir, filename+"_text")
        os.makedirs(cache_dir, exist_ok=True)
        raw_dataset = load_dataset("text", data_files=data_file, cache_dir=cache_dir, keep_in_memory=False)
        print(f"{file} has been loaded")
        tokenized_dataset = raw_dataset.map( #  调用 `raw_dataset.map()` 方法，通过 `tokenize_function` 对数据集进行分词。
            tokenize_function,
            batched=True,
            num_proc=1,
            remove_columns="text",
            load_from_cache_file=True,
            keep_in_memory=False,
            cache_file_names = {k: os.path.join(cache_dir, 'tokenized.arrow') for k in raw_dataset},
            desc="Running tokenizer on dataset",
        )
        grouped_datasets = tokenized_dataset.map( # 对分词后的数据集调用 `group_texts` 函数，将文本分组为固定大小的块。
            group_texts,
            batched=True,
            num_proc=1,
            load_from_cache_file=True,
            keep_in_memory=False,
            cache_file_names = {k: os.path.join(cache_dir, 'grouped.arrow') for k in tokenized_dataset},
            desc=f"Grouping texts in chunks of {block_size}",
        )
        processed_dataset = grouped_datasets
        processed_dataset.save_to_disk(cache_path)
    if idx == 0:
        lm_datasets = processed_dataset['train']
    else:
        assert lm_datasets.features.type == processed_dataset["train"].features.type
        lm_datasets = concatenate_datasets([lm_datasets, processed_dataset["train"]])

lm_datasets = lm_datasets.train_test_split(test_size = 0.2)

if True:
    max_train_samples = None
    train_dataset = lm_datasets['train']
    if max_train_samples is not None:
        max_train_samples = min(len(train_dataset), max_train_samples)
        train_dataset = train_dataset.select(range(max_train_samples))
    print(f"Num train_samples  {len(train_dataset)}")
    print("training example:")
    print(tokenizer.decode(train_dataset[0]['input_ids']))
if True:
    max_eval_samples = None
    eval_dataset = lm_datasets["test"]
    if max_eval_samples is not None:
        max_eval_samples = min(len(eval_dataset), max_eval_samples)
        eval_dataset = eval_dataset.select(range(max_eval_samples))
    print(f"Num eval_samples  {len(eval_dataset)}")
    print("training example:")
    print(tokenizer.decode(eval_dataset[0]['input_ids']))


### 注意
- 数据预处理过程通常涉及大量的数据加载和处理，这可能会导致内存消耗极高。在这段代码中，`load_dataset` 和 `.map()` 函数都提供了 `keep_in_memory` 参数来控制是否将数据保存在内存中，缓存功能的利用也是一种优化手段。
- 数据集的合并需要确保不同来源的数据集具有相同的特征类型。这段代码通过断言 `lm_datasets.features.type == processed_dataset["train"].features.type` 来保证数据集兼容性。
- 使用 `concaneted_examples` 和循环切分技术来处理数据并非总是直观的，特别是在维护索引范围时容易出错。这需要对数据结构和切片技术有较深入的了解。

## 加载和配置语言模型

### 配置BitsAndBytes
在这里，`BitsAndBytesConfig`对象被部分配置。虽然注释掉了一些选项，但`bnb_4bit_compute_dtype`被设定为`torch.float16`，这个配置通常用于调整量化模型的计算精度。

In [ ]:
bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

### 加载模型
这段代码用于从Hugging Face的模型库中加载指定名称的模型，并自动将其分配到合适的设备（如GPU或CPU）上。`AutoModelForCausalLM`是一个用于加载因果语言模型的类。

- `model.config.use_cache = False`: 此行代码禁用模型的缓存功能。在某些情况下，禁用缓存可能有助于减少内存负担或者避免不必要的缓存操作，尤其是在调试和某些推理任务中。

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
#     quantization_config = bnb_config
#     load_in_8bit = True
)

# LLaMA语言模型的初始化与实例化

## 代码作用

### LLaMA配置对象的创建

这段代码通过`LlamaConfig`函数创建了一个配置对象，`vocab_size`参数被设置为32000，表示词汇表的大小。`hidden_size`参数设置为256，表示隐藏层的维度大小。而`num_hidden_layers`设置为2，表示网络中隐藏层的数量。`num_attention_heads`参数为4，表示用于self-attention的头数。这些参数共同决定了模型的架构。

### LLaMA模型的实例化

使用`LlamaForCausalLM(config)`函数，根据上述配置对象来实例化LLaMA语言模型。LLaMA是一种专门用于生成文本的转换器模型，在这里被用作因果语言建模任务。

### 模型输出

通过`print(model)`，代码展示了被初始化后的LLaMA模型的详细结构和参数信息，这有助于检查模型是否按照期望进行配置。

## 重要函数解读

### LlamaConfig

```python
"""
LlamaConfig(vocab_size, hidden_size, num_hidden_layers, num_attention_heads)

LlamaConfig是用于初始化LLaMA模型的配置对象。它规定模型的关键参数如词汇表大小（vocab_size）、隐藏层大小（hidden_size）、隐藏层数（num_hidden_layers）以及注意力头数（num_attention_heads）。
"""
```

### LlamaForCausalLM

```python
"""
LlamaForCausalLM(config)

根据提供的配置对象实例化LLaMA模型，特别适用于因果语言建模任务。通过载入配置对象，模型在初始化时遵循定义的架构设置。
"""
```

## 代码难点

### 正确的配置对象创建

理解如何根据语言模型的需求来准确设置`LlamaConfig`的参数，如词汇表大小和隐藏层参数，是使用这种模型的基础。如果配置不当，可能导致模型性能不佳或内存溢出。

### 模型实例化

当通过传入配置对象来实例化模型时，确保配置与模型预期吻合十分重要。错误的配置可能导致模型不符合需求，或者在训练与推理阶段出现错误。

### 模型结构输出

解析`print(model)`的输出需要对Transformer模型架构有一定了解。理解这个输出有助于验证实例化的模型是否符合需求，并帮助调试可能的错误。

In [ ]:
# # my LLaMA

# from transformers import LlamaModel, LlamaConfig
# from transformers import AutoTokenizer, LlamaForCausalLM

# config = LlamaConfig( vocab_size = 32000, hidden_size = 256,
#                             num_hidden_layers = 2, num_attention_heads = 4)
# model = LlamaForCausalLM(config)
# config = model
# print(model)


## 模型词汇表大小与标记器兼容性检查


### 检查模型与标记器配置的有效性

代码的主要作用是检查模型的词汇表大小与标记器的大小之间的兼容性是否符合预期的配置组合。如果模型的词汇表大小和标记器的大小不匹配预设的配置组合，则抛出一个错误提示用户检查配置。

在代码中，模型的词汇表大小是通过 `model.get_output_embeddings().weight.size(0)` 获取的，而标记器的大小是通过 `len(tokenizer)` 获取的。代码检查了以下几种有效配置：
- 模型词汇表大小为 32000 且标记器大小为 49953。
- 模型词汇表大小和标记器大小均为 32000。
- 模型词汇表大小和标记器大小均为 49953。
- 模型词汇表大小和标记器大小均为 49954。

如果配置不匹配，代码会抛出一个 `ValueError` 提示，说明无效的配置组合，并给出预设的有效组合。

### 注意

- 代码的难点在于模型词汇表大小与标记器大小之间的复杂兼容性配置。在不同的预训练和继续训练阶段，模型的词汇表大小和标记器大小需要严格匹配特定的组合，以确保模型能够正确地处理输入数据。如果一个不匹配的组合被使用，可能导致无法正确地处理输入数据或训练不稳定。因此，理解不同训练阶段对应的正确组合配置是一个关键。
- 另一个难点是理解如何动态调整模型的词汇表大小，并确保调整后的模型仍然能够正常工作。调用 `model.resize_token_embeddings(len(tokenizer))` 方法调整模型的词汇表大小，需要确保模型的其他部分能够正确适应这种调整，尤其是在继续训练的情境下。理解和调试这样的调整需要对模型架构有深入的了解。

In [ ]:
model_vocab_size = model.get_output_embeddings().weight.size(0) # 获取模型的输出嵌入层。
if not (
  (model_vocab_size==32000 and len(tokenizer)==49953) or \
  (model_vocab_size==32000 and len(tokenizer)==32000) or \
  (model_vocab_size==49953 and len(tokenizer)==49953) or \
  (model_vocab_size==49954 and len(tokenizer)==49954)
):
    raise ValueError(
        f"The combination of base model (size: {model_vocab_size}) and tokenizer (size: {len(tokenizer)}) is not a valid configuration. Please check our project wiki for further information. \n"
        "Valid configurations (base model / tokenizer):\n"
        "- Continue pre-training original LLaMA: 32000 / 32000 \n"
        "- Pre-training Chinese LLaMA based on original LLaMA: 32000 / 49953 \n"
        "- Continue pre-training Chinese LLaMA: 49953 / 49953 \n"
        "- Continue pre-training Chinese Alpaca: 49954 / 49954 \n")

### 调整模型词汇表大小

如果有效性检查通过，代码会调用 `model.resize_token_embeddings(len(tokenizer))` 方法来调整模型的词汇表大小以匹配标记器的大小。之后，代码会打印出更新后的模型信息。


In [ ]:
model.resize_token_embeddings(len(tokenizer))
print('resize_token_embeddings:')
print(model)

# LoRA配置与模型训练代码解读

### 配置可训练模块
- `trainable`：包含需要进行LoRA训练的模块（`gate_proj`, `down_proj`, `up_proj`）。随后通过`split(',')`方法将字符串转换为列表并存储在`target_modules`中，为后续配置提供了基础。

### 模块保存配置
- `modules_to_save`：定义了训练过程中需要保存的模型模块（`embed_tokens`, `lm_head`）。同样通过`split(',')`方法将其转换为列表。

In [ ]:
trainable = 'gate_proj,down_proj,up_proj'
modules_to_save = 'embed_tokens,lm_head'
lora_rank = 4
lora_dropout = 0.05
lora_alpha = 32.0
target_modules = trainable.split(',')
modules_to_save = modules_to_save
if modules_to_save is not None:
    modules_to_save = modules_to_save.split(',')

In [ ]:
# trainable = 'q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj'
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # 指定任务类型，例如自然语言生成任务。
    target_modules=target_modules, # 需要应用LoRA的模型模块名单。
    inference_mode=False, # 指定是否为推理模式，False表示训练模式。
    r=lora_rank, # LoRA分解矩阵的秩。
    lora_alpha=lora_alpha, # LoRA缩放因子。
    bias="none", # 指定是否应用偏置。
    lora_dropout=lora_dropout,  # 应用到LoRA层的丢弃率。
    modules_to_save=modules_to_save) # 指定哪些模块在保存时需要特殊处理。



### 配置生成与模型整合
使用`LoraConfig`生成的配置完成后，调用`get_peft_model`函数将LoRA配置应用到目标模型，并输出更新后的模型。
```python
"""
    get_peft_model函数
    :param model: 初始的深度学习模型。
    :param peft_config: 应用的LoRA配置对象。
    :returns: 更新后的可学习参数的深度学习模型。
"""
```

In [ ]:
model = get_peft_model(model, peft_config)
print(model)

### LoRA配置的超参数选择
决定LoRA rank（`lora_rank`）、LoRA alpha（`lora_alpha`）、和dropout（`lora_dropout`）等超参数的值需要一定的经验和实验支撑。这些参数直接关系到模型的收敛速度和效果，需要仔细调试。

这段代码的核心在于将LoRA（低秩适应）方法应用于语言模型的特定模块，从而实现高效且有效的模型训练。对关键配置的理解和参数的选择是难点所在。

# TrainingArguments的核心功能

`TrainingArguments` 是一个用于配置训练过程的参数集合类。通过设置这些参数，可以控制模型训练的详细过程，如优化器设置、日志记录、检查点保存、学习率调度等。

- 配置训练过程中的大多数超参数。
- 管理模型的训练细节，如学习率调度、评估和日志记录策略。
- 提供良好的默认设置以便轻松开始训练。

### 参数的相对复杂性

`TrainingArguments` 包含许多参数，每个参数的相互作用可能会影响最终的训练效果。对于初学者，理解每个参数的作用以及如何配置它们以获得最佳结果可能会有一定难度。

### 混合精度与设备选择

混合精度训练（fp16, bf16）以及设备选择（比如使用 GPU、TPU、MPS）对于训练效率和模型性能有很大的影响。然而，不同硬件支持度差异可能意味着需要适应不同的设置，这可能为跨平台和跨设备部署增加复杂性。

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    adafactor=False, # 控制是否使用 Adafactor 优化器（默认为 False）。
    adam_beta1=0.9, # Adam 优化器的 beta 参数，分别为一阶和二阶矩估计的指数衰减率。
    adam_beta2=0.999, # Adam 优化器用于数值稳定的 epsilon。
    weight_decay=0.01,  # 权重衰减系数，通常用于防止过拟合。
    adam_epsilon=1e-08,
    bf16=False, # 控制是否使用混合精度训练来节省显存和加速训练。
    bf16_full_eval=False,
    max_steps=100, # 最大训练步数。
    num_train_epochs=1.0, # 训练的总轮数。
    deepspeed=None,
    do_eval=False, # 指定是否进行评估过程。
    do_train=True, # 指定是否进行训练过程。
    eval_delay=0,
    eval_steps=100, #  评估所需的步数间隔。
    evaluation_strategy='steps', # 设置评估的策略（如 'steps' 每隔若干步评估一次）。
    fp16=False,
    logging_first_step=True,
    logging_steps=10, # 日志记录相关参数，确定记录日志的频率与策略。
    logging_strategy='steps',
    # 调整学习率
    learning_rate=0.0002, # 设置初始学习率。
    lr_scheduler_type='cosine', # 学习率调度器类型，比如 'cosine' 表示使用余弦退火。
    warmup_ratio=0.05, # 学习率预热参数，决定模型在训练初期逐渐增大学习率。
    # 批次与积累
    per_device_eval_batch_size=4, #  每个设备的评估批次大小。
    per_device_train_batch_size=4, # 每个设备的训练批次大小。
    gradient_accumulation_steps=8, # 梯度积累步数，对于内存有限制的环境中有作用，能在多个步骤后进行反向传播。
    output_dir='./../../data', # 模型输出保存的目录。
    # 检查点保存设置
    save_on_each_node=False,
    save_safetensors=False,
    save_steps=200, # 保存模型检查点的步数间隔。
    save_strategy='steps', #  保存策略（如 'steps' 表示每隔若干步保存一次）。
    save_total_limit=3, #  限制保存的检查点数量。这能够防止磁盘被占满。
    seed=13218, # 整个训练过程的随机种子，可以使实验更具可重复性。
    use_mps_device=True, #mac # 设置是否在 Mac 上使用 MPS（Metal Performance Shaders）设备进行加速。
    warmup_steps=0,
)


## Trainer的初始化

这段代码主要是初始化一个 `Trainer` 对象。`Trainer` 是 Hugging Face 的 Transformers 库中用于简化训练过程的核心类，负责整合模型、训练参数、数据集等信息，并执行训练和评估操作。

### 自定义数据整理器（Data Collator）

`fault_tolerance_data_collator` 是一个数据整理器，它负责将数据样本整理成可以输入模型的批次。在处理不规则输入、实现动态批量大小等情况下，自定义数据整理器可能会比较复杂。

### 性能评估指标的预处理

`preprocess_logits_for_metrics` 函数在评估过程中负责对模型输出进行预处理，以保证评估指标计算的准确性。这个过程通常涉及对模型输出的转换，例如转换为概率分布或应用某种聚合方法。

### 回调函数的实现

`SavePeftModelCallback` 是一个回调函数，实现过程需要确保在训练过程中合适的时点保存模型。这要求对训练过程中的时间点（如 epoch 结束或每个训练步结束）有清晰的理解，以避免遗漏或频繁保存影响性能。

In [ ]:
trainer = Trainer(
        model=model, # 指定要训练的模型。
        args=training_args, # 提供了训练的配置选项，比如学习率、训练轮数等。
        train_dataset=train_dataset , # 训练所用的数据集。
        eval_dataset=eval_dataset if training_args.do_eval else None, #  如果配置允许，则指定用于评估的数据集。
        tokenizer=tokenizer, # 在数据处理过程中使用的分词器。
        data_collator=fault_tolerance_data_collator, # 数据整理器，用于在训练时将样本整理为批次。
        compute_metrics=compute_metrics, # 用于计算评估指标的函数。
        preprocess_logits_for_metrics=preprocess_logits_for_metrics #  用于在计算指标前预处理模型输出的逻辑回归函数。
    )

### 添加回调函数

`trainer.add_callback(SavePeftModelCallback)` 这一行的作用是为 `Trainer` 对象添加一个名为 `SavePeftModelCallback` 的回调函数。回调函数用于在训练的特定阶段（如每个 epoch 结束时）自动执行特定操作。在此上下文中，可能是用于在训练期间保存模型的进度。

In [ ]:

trainer.add_callback(SavePeftModelCallback)

## 模型训练

In [ ]:
do_train = True
do_eval = False

In [ ]:
if do_train:
    train_result = trainer.train() # 调用了训练方法，该方法负责开始模型的训练过程，并返回训练结果。
    metrics = train_result.metrics # 从训练结果中提取评估指标。
    max_train_samples = len(train_dataset) # 计算训练数据集的总样本数。
    metrics["train_samples"] = min(max_train_samples, len(train_dataset)) # 记录训练过程中使用的样本数，确保不会超出数据集的最大样本数。

    trainer.log_metrics("train", metrics) # 记录当前训练阶段的评估指标。
    trainer.save_metrics("train", metrics) # 保存训练阶段的评估指标以供后续分析。
    trainer.save_state() # 保存训练后的状态，这可能包括模型权重、优化器状态等。

In [ ]:
if do_eval:
    metrics = trainer.evaluate() # 调用评估方法以测量模型在验证集上的表现，并返回评估结果。

    max_eval_samples = len(eval_dataset) # 计算评估数据集的总样本数。
    metrics["eval_samples"] = min(max_eval_samples, len(eval_dataset)) # 记录评估过程中使用的样本数。
    try:
        perplexity = math.exp(metrics["eval_loss"])
    except OverflowError:
        perplexity = float("inf")
    metrics["perplexity"] = perplexity #混淆度如何计算 llm

    trainer.log_metrics("eval", metrics) # 记录当前评估阶段的评估指标。
    trainer.save_metrics("eval", metrics) # 保存评估阶段的评估指标。

## 参考资料
1. [Tutorial-Chinese-LLaMA-Alpaca](https://github.com/Flatheadman/Tutorial-Chinese-LLaMA-Alpaca/tree/726fbd43142c4d65daadaa49539adc774cb51516)